In [ ]:
import importlib
spec = importlib.util.find_spec('xgboost')
if spec is None:
    print('xgboost not installed - skipping notebook')
else:
    import json
    from pathlib import Path
    import numpy as np
    import matplotlib.pyplot as plt
    from maneuvers.data.loader import generate_maneuvers_dataset
    from maneuvers.preprocessing import compute_features_from_sequence
    from maneuvers.classify import train_with_grid_search

    DATA_DIR = Path('examples/datasets/maneuvers_small')
    if not DATA_DIR.exists():
        generate_maneuvers_dataset(DATA_DIR, seed=0)

    manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
    # build aggregated feature table (one per sequence)
    X = []
    y = []
    for entry in manifest:
        path = Path(entry['file'])
        from maneuvers.data.loader import from_csv
        seq = from_csv(path)
        feats = compute_features_from_sequence(seq)
        # simple aggregation: mean of features
        X.append(feats[['accel_mag','accel_smooth','gyro_mag']].mean().values)
        y.append(entry['segments'][0][2] if entry['segments'] else 'none')

    X = np.vstack(X)
    out = train_with_grid_search(X, y, model_type='xgb', cv=3)
    print('Best params:', out.get('best_params'))
    # quick confusion matrix on training set
    preds = out['model'].predict(X)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(preds, y, labels=np.unique(y))
    print('Confusion matrix shape:', cm.shape)
    plt.figure(figsize=(6, 4))
    plt.title('Confusion matrix (rows=pred, cols=true)')
    plt.imshow(cm, cmap='Blues')
    plt.colorbar()
    plt.tight_layout()
    plt.show()